# Fraud detection – CatBoost (best submission, public PR-AUC 0.19586)
One model, start to finish: **CatBoost**, a gradient-boosted decision-tree classifier, on 24 features, averaged over 3 random seeds.

| | |
|---|---|
| **Task** | Predict the probability that a card transaction is fraud (`fraud` = 1) for 12,000 test transactions |
| **Metric** | PR-AUC (average precision), plus F1 and Recall at a chosen cut-off |
| **Training data** | 20,000 transactions, 353 frauds (1.77%) |
| **Out-of-fold CV PR-AUC** | 0.238 (5-fold; a random ranker scores ≈ 0.018) |
| **Public leaderboard PR-AUC** | **0.19586** |
| **Output** | `results/catboost/submission_best_catboost.csv`: identical to the 0.19586 upload (checked in the last cell) |

**Steps:** load → clean → engineer features → define the model → cross-validate → choose the flag cut-off → train on all data → write and verify the submission. Each step starts with a short "why".

Runs locally (reads `../data/`) or in Colab (asks for the two CSV files). About 3 minutes on a laptop CPU.

## 1. Setup and data
`THREADS = 3` is fixed on purpose. CatBoost splits the work into one block per thread, so **the thread count changes the fitted trees**: the same model with 3 threads vs all cores differs by up to 0.11 in individual probabilities. Fixing it makes the output reproducible on any machine, including Colab. It is the setting the 0.19586 submission was made with.

In [1]:
%pip install -q catboost scikit-learn pandas numpy
import os, sys, hashlib
import numpy as np
import pandas as pd

THREADS = 3                                        # fixed: CatBoost's result depends on the thread count (see above)
ROOT = ".." if os.path.isdir("../data") else "."
RESULTS = f"{ROOT}/results/catboost"
os.makedirs(RESULTS, exist_ok=True)

# Read from the repo's data/ folder if it exists; otherwise open an upload button (Colab)
def load_csv(name):
    for path in [f"../data/{name}", f"data/{name}", name]:
        if os.path.exists(path):
            return pd.read_csv(path)
    from google.colab import files
    print(f"Upload: {name}")
    return pd.read_csv(list(files.upload().keys())[0])

df = load_csv("Track 2 Training Dataset.csv")
test_df = load_csv("Track 2 Testing Dataset.csv")
print("Train:", df.shape, " Test:", test_df.shape, f" fraud rate {df.fraud.mean():.2%}")

Note: you may need to restart the kernel to use updated packages.


Train: (20000, 12)  Test: (12000, 11)  fraud rate 1.76%


## 2. Cleaning (robust to messy new data)
The current files are clean, so this step changes nothing for them. It exists because the final ranking uses **unseen data** that may be messier:
- **Columns are selected by name;** a missing column is added as all-blank.
- **Numbers are parsed defensively:** text and infinities become blank.
- **Impossible values become blank** instead of being trusted: negative amounts, spend, counts or ages; hours outside 0–23; `new_device` other than 0/1.
- **The 1-hour count is capped at the 24-hour count,** since it can't logically exceed it.
- **Category spelling is normalised** to the training spelling (`" sg"` → `"SG"`); levels never seen in training become blank, and later `"Unknown"`.

Blanks are left as blanks (no zero-filling): CatBoost handles missing numbers natively, and categorical blanks get their own level.

In [2]:
target_column = "fraud"
CATS = ["merchant_category", "country", "transaction_channel"]
RAW_COLS = [c for c in df.columns if c not in ("id", target_column)]
NUMERIC_RAW = [c for c in RAW_COLS if c not in CATS]
NON_NEGATIVE = ["transaction_amount", "spend_last_24h", "transactions_last_1h", "transactions_last_24h", "account_age"]
# Training spelling of every category level, keyed by its trimmed lower-case form (e.g. " sg" -> "SG")
CAT_LEVELS = {c: {str(v).strip().lower(): v for v in df[c].dropna().unique()} for c in CATS}

def sanitise(d):
    d = d.reindex(columns=RAW_COLS).copy()                                 # select columns by name; missing ones become NaN
    for c in NUMERIC_RAW:
        d[c] = pd.to_numeric(d[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
    for c in NON_NEGATIVE:
        d[c] = d[c].where(d[c] >= 0)                                       # negative -> NaN
    d["transaction_hour"] = d.transaction_hour.where(d.transaction_hour.between(0, 23))
    d["new_device"] = d.new_device.where(d.new_device.isin([0, 1]))
    over = d.transactions_last_1h > d.transactions_last_24h                # 1h count can't exceed the 24h count
    d.loc[over, "transactions_last_1h"] = d.loc[over, "transactions_last_24h"]
    for c in CATS:                                                         # normalise spelling; unseen levels -> NaN -> "Unknown"
        d[c] = d[c].astype("string").str.strip().str.lower().map(CAT_LEVELS[c]).astype(object)
    return d

## 3. Features
Each feature uses only that row's own values: no labels and no information from other rows, so there is no leakage. The 24 features fall into four groups.

| Group | Features | Why |
|---|---|---|
| **Raw columns (10)** | amount, hour, merchant, country, channel, 24h count, 24h spend, account age, new device, 1h count | The strongest signals live here: new device (7.5% fraud vs 1.2%), ≥ 5 transactions in an hour (23–30%), the top amount decile (5.6%), accounts under ~70 days old (4.6%). |
| **Smooth transforms** | `log_amt`, `log_spend`, `log_age`, `hour_sin`, `hour_cos` | Amounts and ages are heavily skewed, and logs give the trees evenly spread split points. Sine/cosine make 23:00 and 00:00 neighbours. |
| **Behaviour relative to the account itself** | `amt_to_avg_ticket` (this amount vs the account's average over 24h), `share_1h` (share of the last 24h's transactions that fell in the last hour) | Fraud is a transaction that is unusual *for that account*, which a raw amount can't express. |
| **Risk indicators from the EDA** | `night` (23:00–04:59), `burst` (≥ 4 in 1h), `high24` (≥ 11 in 24h), `young` (≤ 30 days), `risky_merchant` (luxury, cash transfer, electronics), `night_x_device`, `n_flags` (count of fired indicators) | They encode the EDA findings directly. Combinations are where fraud concentrates (pairs reach 38–40% fraud), and `n_flags` / `night_x_device` hand the shallow trees those combinations ready-made. |

**Categoricals** are passed to CatBoost as text; blanks become `"Unknown"`. CatBoost turns each level into a fraud-rate statistic computed only from *earlier* rows in a random order ("ordered target statistics"). That is its built-in protection against target leakage, so no manual target encoding is needed.

The test set gets exactly the same transformation and the training set's category lists.

In [3]:
def engineer(d):
    d = sanitise(d)
    d["log_amt"] = np.log1p(d.transaction_amount)
    d["log_spend"] = np.log1p(d.spend_last_24h)
    d["log_age"] = np.log1p(d.account_age)
    d["hour_sin"] = np.sin(2 * np.pi * d.transaction_hour / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d.transaction_hour / 24)
    d["night"] = d.transaction_hour.isin([23, 0, 1, 2, 3, 4]).astype(int)
    d["burst"] = (d.transactions_last_1h >= 4).astype(float)
    d["high24"] = (d.transactions_last_24h >= 11).astype(float)
    d["young"] = (d.account_age <= 30).astype(float)
    d["amt_to_avg_ticket"] = d.log_amt - np.log1p(d.spend_last_24h / d.transactions_last_24h.clip(lower=1))
    d["share_1h"] = d.transactions_last_1h / d.transactions_last_24h.clip(lower=1)
    d["risky_merchant"] = d.merchant_category.isin(["luxury", "cash_transfer", "electronics"]).astype(int)
    d["night_x_device"] = d.night * d.new_device
    d["n_flags"] = d[["burst", "high24", "young", "new_device", "night", "risky_merchant"]].fillna(0).sum(axis=1)
    for c in CATS:
        d[c] = d[c].fillna("Unknown").astype("category")
    return d

# The 24 model features, in the exact order used for the 0.19586 submission (CatBoost can depend on column order)
FEATURES = ["transaction_amount", "transaction_hour", "merchant_category", "country", "transaction_channel",
            "transactions_last_24h", "spend_last_24h", "account_age", "new_device", "transactions_last_1h",
            "log_amt", "log_spend", "log_age", "hour_sin", "hour_cos", "night", "burst", "high24", "young",
            "amt_to_avg_ticket", "share_1h", "risky_merchant", "night_x_device", "n_flags"]

X = engineer(df.drop(columns=["id", target_column]))[FEATURES]
y = df[target_column].values
X_test = engineer(test_df.drop(columns=["id"]))
for c in CATS:                                    # test categories use the training category lists
    X_test[c] = pd.Categorical(X_test[c].astype(str), categories=X[c].cat.categories)
X_test = X_test[FEATURES]

def as_catboost(d):                               # CatBoost takes categoricals as text
    return d.astype({c: str for c in CATS})

print(f"{len(FEATURES)} features; train {X.shape}, test {X_test.shape}")

24 features; train (20000, 24), test (12000, 24)


## 4. Model: CatBoost
**Why CatBoost:**
- **It held up best where it counts.** On the public leaderboard it beat every alternative we submitted: LightGBM 0.190, TabPFN 0.188, random forest blends, logistic blends (0.183), monotone-constrained CatBoost (~0.15).
- **It finds combinations of signals by itself,** and that's where fraud concentrates in this data.
- **Native categoricals with leakage-safe encoding,** and native handling of missing values.

**Settings and why:**

| Setting | Value | Why |
|---|---|---|
| `depth` | 4 | Small trees: each can combine up to 4 signals, enough for the fraud combinations, but too shallow to memorise individual rows among only 353 frauds. Depth 3 and 5 both scored lower in CV. |
| `learning_rate` × `iterations` | 0.03 × 800 | Many small steps generalise better than a few large ones. |
| `l2_leaf_reg` | 10 | Strong shrinkage of leaf values against noise from 353 frauds. 20–30 scored lower. |
| class weights | none | Weighting frauds up (`scale_pos_weight`, `SqrtBalanced`) lowered CV PR-AUC by 0.016–0.04. PR-AUC only needs a good *ranking*, and weighting distorts it. Recall comes from the cut-off instead (step 6). |
| seeds | 0, 1, 2, averaged | Averaging three fits reduces the randomness of a single fit. |
| `thread_count` | 3 | Reproducibility (see step 1). |

Searches over depth, L2, learning rate, feature subsampling, one-hot categoricals, coarser borders and Depthwise growth found nothing better than these settings.

In [4]:
from catboost import CatBoostClassifier

SEEDS = [0, 1, 2]

def make_catboost(seed):
    return CatBoostClassifier(iterations=800, depth=4, learning_rate=0.03, l2_leaf_reg=10, cat_features=CATS,
                              random_seed=seed, verbose=0, allow_writing_files=False, thread_count=THREADS)

def fit_predict(X_fit, y_fit, X_score, seeds):
    """Average fraud probability of one CatBoost per seed."""
    return np.mean([make_catboost(s).fit(as_catboost(X_fit), y_fit).predict_proba(as_catboost(X_score))[:, 1]
                    for s in seeds], axis=0)

## 5. Cross-validation
**5-fold stratified CV** (every fold keeps the 1.77% fraud rate). Each training row is scored once by a model that never saw it, giving out-of-fold predictions for all 20,000 rows. Two seeds per fold keep the run short. There's no timestamp and no account id, so a time-based or grouped split isn't possible.

**What CV can and can't tell us:**
- It measures how well the model ranks *training-like* data: PR-AUC ≈ 0.24 vs ≈ 0.018 for a random ranker.
- The test set is shifted. It has more new devices (9% → 15%), more bursts (2.3% → 4.4%), larger amounts and 3× more missing values, and those signals are fraud less often there. Every model we built scored ~0.045 lower on the leaderboard than in CV.
- So CV was used to choose *within* this design (settings, features), and the leaderboard to choose *between* designs.

In [5]:
from sklearn.metrics import average_precision_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold

CV_SEEDS = [0, 1]
oof = np.zeros(len(y))
for k, (tr, te) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=0).split(X, y)):
    oof[te] = fit_predict(X.iloc[tr], y[tr], X.iloc[te], CV_SEEDS)
    print(f"fold {k} done", flush=True)
print(f"Out-of-fold PR-AUC: {average_precision_score(y, oof):.4f}   (random ranker ~ {y.mean():.3f})")

fold 0 done


fold 1 done


fold 2 done


fold 3 done


fold 4 done


Out-of-fold PR-AUC: 0.2384   (random ranker ~ 0.018)


## 6. Cut-off for the fraud flags (F1 / Recall)
PR-AUC only uses the *ordering* of the scores, so no cut-off can change it. We checked this on the leaderboard: the same model scored the same with and without rescaling. F1 and Recall, however, need 0/1 flags.

- **Choosing the share:** for every share of rows flagged (0.2% to 6%, in 0.05% steps), build the confusion matrix on the out-of-fold scores and compute precision, recall and F1. Pick the share with the best F1, smoothed over 5 neighbouring shares so a one-off spike isn't chosen.
- **Mapping to 0.5:** most likely the organisers flag rows at 0.5, so scores are mapped monotonically into [0, 1] with that share above 0.5. The order is unchanged, so PR-AUC is unchanged.

In [6]:
def pick_share(score):
    rows = []
    for share in np.arange(0.002, 0.0601, 0.0005):
        flag = (score >= np.quantile(score, 1 - share)).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, flag).ravel()
        rows.append({"share": share, "TP": tp, "FP": fp, "FN": fn,
                     "precision": tp / max(tp + fp, 1), "recall": tp / (tp + fn), "F1": 2 * tp / (2 * tp + fp + fn)})
    cm = pd.DataFrame(rows)
    cm["F1_smooth"] = cm["F1"].rolling(5, center=True, min_periods=1).mean()   # don't pick a one-off noisy spike
    return cm.loc[cm["F1_smooth"].idxmax()]

best = pick_share(oof)
share = best["share"]
print(f"Flag the top {share:.2%} of rows: precision {best.precision:.3f}, recall {best.recall:.3f}, F1 {best.F1:.3f} "
      f"(TP {best.TP:.0f}, FP {best.FP:.0f}, FN {best.FN:.0f})")
pd.DataFrame([{"model": "catboost", "PR-AUC": average_precision_score(y, oof), "flag share": share, "precision": best.precision,
               "recall": best.recall, "F1": best.F1}]).round(4).to_csv(f"{RESULTS}/cv_metrics_best_catboost.csv", index=False)

Flag the top 1.65% of rows: precision 0.315, recall 0.295, F1 0.305 (TP 104, FP 226, FN 249)


## 7. Final model and submission
Train on all 20,000 rows (3 seeds), score the 12,000 test rows, map to [0, 1] with the cut-off at 0.5, and check the file: right length, unique ids in the sample file's order, and every prediction in [0, 1].

In [7]:
def to_submission_score(score, thr):
    """Monotone map to [0,1] that puts the flag cut-off at 0.5 (order, and so PR-AUC, unchanged)."""
    score = np.asarray(score, dtype=float)
    lo, hi = score.min(), score.max()
    below = 0.49 * (score - lo) / max(thr - lo, 1e-12)
    above = 0.51 + 0.49 * (score - thr) / max(hi - thr, 1e-12)
    return np.clip(np.where(score >= thr, above, below), 0, 1)

score = fit_predict(X, y, X_test, SEEDS)
score = np.where(np.isfinite(score), score, np.nanmedian(score))           # defensive: a missing score gets the median
sub = pd.DataFrame({"id": test_df["id"].values, "prediction": to_submission_score(score, np.quantile(score, 1 - share))})

assert len(sub) == len(test_df) and sub["id"].is_unique and sub["prediction"].between(0, 1).all()
for path in [f"{ROOT}/format/sample_submission.csv", "sample_submission.csv"]:
    if os.path.exists(path):
        assert (pd.read_csv(path)["id"].values == sub["id"].values).all(), "ids not in sample order"
OUT = f"{RESULTS}/submission_best_catboost.csv"
sub.to_csv(OUT, index=False)
print(f"{OUT}: {len(sub)} rows, flagged (>= 0.5): {(sub.prediction >= 0.5).sum()}")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(OUT)

../results/catboost/submission_best_catboost.csv: 12000 rows, flagged (>= 0.5): 198


## 8. Reproducibility check
The file must be byte-identical to the submission that scored 0.19586 (`results/leaderboard_uploads/01_catboost_3seeds_FINAL.csv`, MD5 `d666afc9c1454c2933b2aca8e21933fb`).

In [8]:
EXPECTED_MD5 = "d666afc9c1454c2933b2aca8e21933fb"
md5 = hashlib.md5(open(OUT, "rb").read()).hexdigest()
print("MD5:", md5, "-> identical to the 0.19586 submission" if md5 == EXPECTED_MD5 else "-> DIFFERENT from the 0.19586 submission")
ref = f"{ROOT}/results/leaderboard_uploads/01_catboost_3seeds_FINAL.csv"
if os.path.exists(ref):
    r = pd.read_csv(ref)
    print("max |difference| vs the uploaded file:", float(np.abs(r.prediction.values - sub.prediction.values).max()))

MD5: d666afc9c1454c2933b2aca8e21933fb -> identical to the 0.19586 submission
max |difference| vs the uploaded file: 1.1102230246251565e-16


## 8b. Save the model bundle
The three fitted CatBoost models, the flag share and every constant the scoring code depends on are saved together. This is the exact layout `final_notebook/predict_catboost.ipynb` loads (its first check compares these constants), so `results/catboost/model.pkl` can replace `final_notebook/model.pkl`. `python -m fraudlab train` writes the same bundle.

In [9]:
import joblib

final_models = [make_catboost(s).fit(as_catboost(X), y) for s in SEEDS]
assert np.allclose(np.mean([m.predict_proba(as_catboost(X_test))[:, 1] for m in final_models], axis=0), score)
bundle = {"models": final_models, "features": FEATURES, "cats": CATS, "raw_cols": RAW_COLS, "non_negative": NON_NEGATIVE,
          "cat_levels": CAT_LEVELS, "categories": {c: list(X[c].cat.categories) for c in CATS}, "share": float(share)}
joblib.dump(bundle, f"{RESULTS}/model.pkl")
print(f"{RESULTS}/model.pkl: {len(final_models)} models, flag share {share:.2%}")


../results/catboost/model.pkl: 3 models, flag share 1.65%


## 9. What else was tried, and why this model was kept
All alternatives were judged on the public leaderboard (12,000 test rows, ~212 frauds). The same model moves ±0.004 just from the choice of seeds, so only bigger differences mean anything.

| Alternative | Public PR-AUC | Lesson |
|---|---|---|
| **This model** (CatBoost, 3 seeds) | **0.19586** | – |
| TabPFN + LightGBM + logistic blend | 0.19354 | More models did not help |
| CatBoost 1 seed / 10 seeds | 0.1925 / 0.1921 | Seed count is noise, not a lever |
| LightGBM | 0.1901 | – |
| Shift corrections (flag-excess correction + within-dataset context features) | 0.1893 | Reweighting signals for the test mix didn't help |
| Unlabelled test rows added as "not fraud" | 0.1878 | The test's extra flagged rows are not simply legitimate |
| TabPFN | 0.1875 | – |
| CatBoost + logistic 50/50 | 0.1832 | Logistic regression gives every flag a fixed bonus, which misfires on the shifted test set |
| Monotone-constrained CatBoost | ~0.15 | Forcing "more signal = more risk" everywhere amplified the misfiring flags |

Rejected on cross-validation before any upload:
- class weighting (−0.016 to −0.04);
- target / WoE encoding, missing-value indicators, splines and explicit interactions (all within noise);
- depth 3 / 5 and stronger regularisation;
- relative "context" features (−0.03);
- kNN / neural-net ensembles (−0.06).

**Model assumptions** (for the report):
1. **Rows are independent.** There is no account id or timestamp to model sequences.
2. **The training labels are correct.**
3. **Fraud signals in the test set point the same way as in training.** This is the assumption the train → test shift strains: the same flags are fraud less often in the test set, which is why every model scores ~0.045 lower there than in CV.